<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

----------

<h1 align="left"> Building a RAG System with Document Chunking </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

----------

## Overview

In the demo `Building_a_Simple_RAG_System_from_Scratch.ipynb`, five hand-written sentences were indexed. Each sentence acted as a tiny "document," which kept the pipeline simple but did not look much like real data.

Real corpora are longer. A PDF, Markdown page, or article often will not fit in a generation model's context window, and one giant embedding blurs too many ideas together. The standard fix is **chunking**: split documents into shorter, overlapping passages, embed each passage, and keep a source pointer for citation.

In this lab, rebuild the same RAG pipeline with multi-paragraph documents and a chunking step. The next lab (`Build_a_Research_Paper_Assistant_using_RAG(lab).ipynb`) reuses this pipeline shape on a different corpus.

## Choose your path

Choose one environment. Everything after document loading is the same.

**Path A &mdash; Local (VS Code + Jupyter, recommended).** Add `.txt` or `.md` files to a `./documents/` folder next to this notebook. Budget roughly 2&ndash;4 GB of free RAM for the embedding and generation models on CPU; a GPU is helpful but not required. Using familiar documents makes retrieval and generation easier to judge.

**Path B &mdash; Google Colab.** If your laptop is tight on memory, run the notebook in Colab. Instead of loading local files, pull a long-form dataset such as Wikipedia or arXiv full text.

In Task 1, uncomment one loading path and leave the other commented. The rest of the notebook is shared.

## What you will build

1. Load documents from disk (Path A) or from a Hugging Face dataset (Path B).
2. Write a `chunk_text(text, chunk_size, overlap)` helper.
3. Apply the chunker to every document and remember which document each chunk came from.
4. Embed the chunks with `sentence-transformers`.
5. Build a FAISS index over the chunk embeddings.
6. Write a retrieval function that returns the chunk text **and** its source.
7. Load a FLAN-T5 generation pipeline.
8. Compose the full RAG function so answers use retrieved chunks and print citations.
9. Test the pipeline on at least two queries and inspect what was retrieved.

## Setup

Install dependencies for the environment you are using.

In [1]:


pip install faiss-cpu sentence-transformers transformers datasets==2.14.5

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.1 MB/

In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd
from pathlib import Path

## Task 1 &mdash; Load your raw documents

Create a Python list `raw_documents`, where each entry is a dict
`{"id": <str>, "text": <str>}`. `id` is a short citation label, such as a filename, article title, or paper id. `text` is the full document body.

Aim for 3&ndash;20 documents with **at least a few paragraphs each**. Very short inputs will not show why chunking matters.

In [3]:

# ============================================================
# Path B (Colab): pull a long-form text dataset from Hugging Face
# ============================================================
# Pick one of these. Wikipedia is more readable; arXiv full-text is
# denser. For arXiv, use the full article, not just the abstract.
#
from datasets import load_dataset

ds = load_dataset("wikipedia", "20220301.simple", split="train[:25]")
raw_documents = [{"id": row["title"], "text": row["text"]} for row in ds]

print(f"Loaded {len(raw_documents)} documents")
print(f"First doc id: {raw_documents[0]['id']}")
print(f"First doc length (chars): {len(raw_documents[0]['text'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Downloading:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/235M [00:00<?, ?B/s]

Loaded 25 documents
First doc id: April
First doc length (chars): 15966


## Task 2 &mdash; Write a chunker

A chunker turns one long string into shorter passages. Two parameters matter most:

- `chunk_size` &mdash; characters per chunk. A good starting point for MiniLM-style embeddings is 500&ndash;1000 characters.
- `overlap` &mdash; characters repeated from the end of one chunk at the start of the next. Overlap helps when an answer would otherwise be split across a boundary. Start with 10&ndash;20% of `chunk_size`.

For this lab, chunk by **characters**. Production systems usually chunk by tokens or sentences to avoid cutting words in half, but the core idea is the same.

In [4]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> list[str]:
    if overlap >= chunk_size:
        raise ValueError(f"overlap ({overlap}) must be less than chunk_size ({chunk_size})")

    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step

    return chunks


# Sanity checks
assert chunk_text("abcdef", chunk_size=10, overlap=2) == ["abcdef"]
assert chunk_text("abcdefghij", chunk_size=4, overlap=1) == ["abcd", "defg", "ghij"]
try:
    chunk_text("hello", chunk_size=5, overlap=5)
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError when overlap >= chunk_size")
print("chunk_text passes basic checks")

chunk_text passes basic checks


## Task 3 &mdash; Apply the chunker and keep provenance

Run `chunk_text` over every raw document and build **two parallel lists**:

- `chunks` &mdash; the chunk strings to embed.
- `chunk_sources` &mdash; matching `(doc_id, chunk_index)` tuples, so when FAISS returns row 42, the original document and chunk number are still known.

Keep this invariant: `chunks[i]` and `chunk_sources[i]` always refer to the same chunk.

In [5]:
chunks: list[str] = []
chunk_sources: list[tuple[str, int]] = []

for doc in raw_documents:
    doc_chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        chunk_sources.append((doc["id"], i))

print(f"Built {len(chunks)} chunks from {len(raw_documents)} documents")
print(f"Example source: {chunk_sources[0]}")
print(f"Example chunk (first 200 chars): {chunks[0][:200]!r}")

Built 223 chunks from 25 documents
Example source: ('April', 0)
Example chunk (first 200 chars): 'April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of week as '


## Task 4 &mdash; Embed the chunks

Use the same `sentence-transformers` model as the demo: `sentence-transformers/all-MiniLM-L6-v2` (384-dim, fast on CPU).

On Colab with a GPU, you can try `all-mpnet-base-v2` (768-dim, more accurate, slower). Stick to MiniLM on Path A unless your laptop can handle more.

Encode all chunks in one `encode` call. Set `normalize_embeddings=True` so you can use inner-product (cosine) search downstream.

In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embeddings shape:", chunk_embeddings.shape)
# Expected: (len(chunks), 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (223, 384)


## Task 5 &mdash; Build a FAISS index over the chunks

Because the embeddings are normalised, use `IndexFlatIP` (inner product on unit vectors equals cosine similarity). `IndexFlatIP` is exact and fine for a few thousand chunks. Larger corpora usually need approximate FAISS indexes such as `IndexIVFFlat` or `IndexIVFPQ`.

In [7]:
dimension = chunk_embeddings.shape[1]  # 384 for MiniLM

index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print(f"FAISS index contains {index.ntotal} chunks.")
# Expected: same number as len(chunks)

FAISS index contains 223 chunks.


## Task 6 &mdash; Retrieval with source attribution

The demo returned text only. Here, return the chunk text **and** its source. That makes each answer traceable to a specific document and chunk.

In [8]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, top_k)

    results = []
    for rank in range(top_k):
        idx = indices[0][rank]
        doc_id, chunk_index = chunk_sources[idx]
        results.append({
            "text": chunks[idx],
            "doc_id": doc_id,
            "chunk_index": chunk_index,
            "score": float(distances[0][rank]),
        })
    return results


# Sanity check
results = retrieve("What month comes after March?", top_k=3)
for r in results:
    print(f"[{r['doc_id']} #{r['chunk_index']}] score={r['score']:.3f}")
    print(r['text'][:200], "...\n")

[April #21] score=0.573
 the Netherlands. Beatrix later also abdicates, on this day in 2013, in favor of her son, King Willem-Alexander of the Netherlands.

Trivia 

 In Western Christianity, there is a bigger likelihood of  ...

[August #0] score=0.565
August (Aug.) is the eighth month of the year in the Gregorian calendar, coming between July and September. It has 31 days. It is named after the Roman emperor Augustus Caesar.

August does not begin  ...

[August #1] score=0.545
 Pompilius about 700 BC. Or, when those two months were moved from the end to the beginning of the year by the decemvirs about 450 BC (Roman writers disagree). In 153 BC January 1 was determined as th ...



## Task 7 &mdash; Generation model

Load a text2text-generation pipeline with `google/flan-t5-base`, the same model used in the demo.

Path A note: `flan-t5-base` is ~990 MB to download and uses around 1&ndash;2 GB of RAM on CPU. If that is too heavy, use `google/flan-t5-small` (~308 MB). Answers may be shorter or less accurate, but the pipeline will still run.

In [9]:
# TODO: build the generation pipeline.
generator = pipeline("text2text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


## Task 8 &mdash; The full RAG pipeline

Put retrieval and generation together. The pipeline should:

1. Retrieve the top-k chunks for the query.
2. Concatenate their text into a `context` string.
3. Build a prompt of the form
   ```
   Context: <context>
   Question: <query>
   Answer:
   ```
4. Run the generator on the prompt.
5. Print the answer **and** citations of the form `[doc_id #chunk_index]` so grounding is easy to check.

In [10]:
def rag_pipeline(query: str, top_k: int = 3, max_new_tokens: int = 120) -> None:
    results = retrieve(query, top_k=top_k)

    context = "\n\n".join(r["text"] for r in results)

    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    response = generator(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"]

    print("Query:", query)
    print()
    print("Retrieved chunks:")
    for r in results:
        print(f"  [{r['doc_id']} #{r['chunk_index']}] (score {r['score']:.3f}) {r['text'][:160]}...")
    print()
    print("Answer:", response)
    print()

## Task 9 &mdash; Test the pipeline

Run two or three questions where the answer is known from the loaded documents. For each one, check three things:

- The retrieved chunks contain the information needed to answer.
- The generated answer is consistent with the retrieved chunks (no hallucinations).
- The cited `doc_id #chunk_index` actually points to the chunk that mentions the fact.

If the chunks look wrong, tune retrieval with a different `chunk_size`, `overlap`, or `top_k`. If the chunks look right but the answer is wrong, try a bigger generator (`flan-t5-large`) or rewrite the prompt.

In [12]:
# TODO: replace these with queries that match the loaded documents.
rag_pipeline("What month comes after March?")
rag_pipeline("How many days does April have?")
rag_pipeline("What is the capital of France?")

Query: What month comes after March?

Retrieved chunks:
  [April #21] (score 0.573)  the Netherlands. Beatrix later also abdicates, on this day in 2013, in favor of her son, King Willem-Alexander of the Netherlands.

Trivia 

 In Western Christ...
  [August #0] (score 0.565) August (Aug.) is the eighth month of the year in the Gregorian calendar, coming between July and September. It has 31 days. It is named after the Roman emperor ...
  [August #1] (score 0.545)  Pompilius about 700 BC. Or, when those two months were moved from the end to the beginning of the year by the decemvirs about 450 BC (Roman writers disagree). ...

Answer: May

Query: How many days does April have?

Retrieved chunks:
  [April #0] (score 0.728) April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.

April a...
  [April #1] (score 0.683) r and on the same day of the week as January in leap years. April ends on the s

## Stretch goals

Try one or two of these after the core pipeline runs cleanly.

1. **Sweep chunk sizes.** Re-run the pipeline with `chunk_size` in `[400, 800, 1600]` and `overlap` set to ~10% of `chunk_size`. For a fixed query, which setting retrieves the most useful chunk, and why?

2. **PDF input (Path A only).** Add `pip install pypdf` and extend the Task 1 loader so `.pdf` files in `./documents/` are read with `pypdf.PdfReader` and the per-page text is concatenated before chunking.

3. **A different FAISS index.** Replace `IndexFlatIP` with `IndexIVFFlat`. Train it before calling `add`, then compare latency on the same query.

4. **Stronger embeddings.** Swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` (768-dim). Re-index and re-run the same query. Did the retrieved chunks change? Is the answer better?

5. **Re-rank.** After FAISS retrieval, re-rank the top-k chunks with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) before passing them to the generator. Cross-encoders are slower than the bi-encoder used for indexing but typically more accurate at ranking the final shortlist.